In [1]:
# 행렬을 이용한 내적연산 -> 벡터의 내적연산
# w1x1 + w2x2 + w3x3...+ b
# step function : 0보다 크면 1, 0보다 작으면 0
# 활성화 함수 : step function, sigmoid function, relu function
# 계단함수가 필요한 이유 : 비선형적인 행동을 해야지만 구역을 나누는 기준선(Decision Boundary)을 만들 수 있다
# 활성화 함수가 없으면 아무리 층을 깊게 쌓아도 결국 y = wx + b 형태의 선형함수 밖에 만들 수 없다.


In [2]:
# step function
def step_function(x):
    return (x >= 0).float()

In [3]:
import torch

x = torch.tensor([
    [0.0, 0.0], 
    [0.0, 1.0], 
    [1.0, 0.0], 
    [1.0, 1.0]
])

w = torch.tensor([[0.5], [0.5]])  # 수정
b = -0.7

z = torch.matmul(x, w) + b
output = step_function(z)
# And gate
for x_val, out_val in zip(x,output):
    print(f'입력 {x_val}, 출력 {out_val}')

입력 tensor([0., 0.]), 출력 tensor([0.])
입력 tensor([0., 1.]), 출력 tensor([0.])
입력 tensor([1., 0.]), 출력 tensor([0.])
입력 tensor([1., 1.]), 출력 tensor([1.])


In [ ]:
# XOR
# 입력이 서로 다를때 만 1을 출력

# 임의로 가중치를 xor
w_xor = torch.tensor([[1.0], [1.0]])
b_xor = -0.5

z = torch.matmul(x,w_xor) + b_xor
output_xor = step_function(z)
for x_val, out_val in zip(x,output_xor):
    corrent_target = int(x_val[0] != x_val[1])
    print(f'입력 {x_val}, 출력 {out_val}, 정답 : {corrent_target}')

입력 tensor([0., 0.]), 출력 tensor([0.]), 정답 : 0
입력 tensor([0., 1.]), 출력 tensor([1.]), 정답 : 1
입력 tensor([1., 0.]), 출력 tensor([1.]), 정답 : 1
입력 tensor([1., 1.]), 출력 tensor([1.]), 정답 : 0


In [ ]:
# 다층 퍼셉트론 : 은닉층(hidden layer)
# 활성화 함수 : Sigmoid, ReLU 같은 파생 활성화 함수 사용

# x = [1.0, 0.0]
# z1 = (w11 * x1) + (w21 * x2) + b1
# z2 = (w12 * x1) + (w22 * x2) + b2
# z = x * w + b

# z의 값에 활성화 함수 시그모이드 씌우면 - 값이 아무리 크거나 작아도 0~1 사이로 압축
# 직선이었던 데이터 공간이 곡면으로 휘어짐

# x
# z = x* w + b
# h = sigmoid(z)
# z2 = h * w2 + b2
# y = sigmoid(z2)
# y_pred = y

# 입력 2 - 은닉층 2개 뉴런 - 출력 1개 뉴런
# x = [1.0, 0.0]
# w1 = [
#     20, -20'
#     20, -20
# ]
# b = [-10, 30]
# w2 = [
#     20
#     20
# ]
# b2 = [-30]
# # 은닉층 선형결합
# z = [1.0, 0.0] @ [20 -20        +[-10,30] = [10,10]
#                   20 -20]
# 시그모이드 [0.9999, 0.9999]
# 출력층 [0.9999,0.9999] @ [20   + (-30) = 39.998 - 30 = 9.998
#                          20]
# 출력 활성화 시그모이드(9.998) = 0.9999999   판정 : 1

In [ ]:
# 활성화 함수의 필요성 : 선형 붕괴(활성화 함수 없이 층을 쌓으면 의미 없음)
# 1층 : h = w1 * x + b1
# 2층 : y = w2 * h + b2 = w2 * (w1 * x + b1) + b2
# (w2 * w1 * x) + (w2 * b1) + b2
# w' = w2* w1, b' = w2 * b1 + b2
# y = w' * x + b' # 활성화 함수 없으면 결국 직선의 형태(선형)

# 활성화 함수는 공간변형 도구 
# 계단 함수는 판단은 가능하지만 학습은 불가능(미분 불가능, 기울기 알 수 없음 or 기울기 0)

In [ ]:
import torch
torch.sigmoid( torch.tensor(0) )


tensor(0.5000)

In [ ]:
#  x  시그모이드 시그모이드 미분
#  0     0.5       0.25  가장 활발하게 학습
# +-2  0.88/0/12   0.10  학습속도 둔화
# +-5  0.00/0.01  0.0067 거의 학습 안됨
# +- 10  1 / 0      0    완전 정지 (뉴런 사망)

In [ ]:
# torch 자동 미분 기능
import torch
x = torch.tensor([0.0], requires_grad=True)
y = torch.sigmoid(x)
y.backward()
print( x.grad )

tensor([0.2500])


In [ ]:
# ReLU
# f(x) = z = max(0, x)
import numpy as np
x = np.array([-5, 0, 2, 6 ,-5])

relu = np.maximum(0,x) # 0이하 0, 이상 x
relu

array([0, 0, 2, 6, 0])

In [ ]:
# ReLU는 미분을 하면 야수구간은 항상 1
# 1 * 1 * 1 * 1 == 1 기울기가 죽지 않고 끝까지 전달
# 입력이 음수면 기울기가 0 => 출력이 비활성화 => 계산중에 불필요한 뉴런 제거하는 효과
# 시그모이드보다 6배 이상 빠르고 안정적으로 학습이 가능


In [ ]:
# 활성화 없이 100층 쌓기
